# Notebook 01 — Intro to LLM Systems (2026)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/01_intro_to_llm_systems_2026.ipynb)

This notebook introduces the systems layer that begins after you choose or finetune an open model. The central shift is simple: a good model is necessary, but product behavior is determined by the runtime around it.

We will stay notebook-native and use small simulations whenever a production runtime would be too heavy for a first-principles lesson.

## What you will learn

- why LLM systems work starts where model selection stops
- the request lifecycle from arrival to post-processing
- why prefill and decode behave differently
- how throughput, latency, reliability, and cost pull against each other
- how cache thinking and observability change system design

## The mental model for Module 1

Treat an LLM request as a pipeline with budgets. Every request spends prompt tokens, decode steps, memory, queue time, and engineering attention. A systems engineer makes those budgets explicit before reaching for a faster runtime.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

In [ ]:
random.seed(1)

NOTEBOOK_ROOT = ARTIFACT_ROOT / "module_01_intro_to_llm_systems"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def min_max_scale(series, larger_is_better=True):
    values = series.astype(float)
    span = values.max() - values.min()
    if span == 0:
        scaled = pd.Series([1.0] * len(values), index=series.index)
    else:
        scaled = (values - values.min()) / span
    if not larger_is_better:
        scaled = 1.0 - scaled
    return scaled.round(3)

print("Artifact root:", NOTEBOOK_ROOT.resolve())

## Step 1: Put system goals next to model goals

Teams often evaluate only answer quality. Systems work adds four more questions:

1. how long does the user wait?
2. how many requests can the runtime absorb?
3. how much memory and cost does each token consume?
4. how visible are failures when load rises or inputs get longer?

In [ ]:
axes = pd.DataFrame([
    {"area": "quality", "core_question": "Is the answer useful and correct?", "failure_if_ignored": "Fast wrong answers ship", "primary_metric": "task score"},
    {"area": "latency", "core_question": "How long until the first and last useful tokens arrive?", "failure_if_ignored": "The product feels slow", "primary_metric": "TTFT and end-to-end ms"},
    {"area": "throughput", "core_question": "How many requests or tokens can we serve per second?", "failure_if_ignored": "Queues explode under load", "primary_metric": "req/s and tok/s"},
    {"area": "memory", "core_question": "Do weights and KV cache fit with headroom?", "failure_if_ignored": "OOMs and thrash", "primary_metric": "peak GB"},
    {"area": "reliability", "core_question": "Can the service keep its contract when reality gets messy?", "failure_if_ignored": "Retries, drops, and surprise regressions", "primary_metric": "success rate and SLO pass rate"},
])
axes

## Step 2: Turn tasks into workloads

A workload is more concrete than a task label. It includes prompt length, output length, concurrency expectations, and reliability targets. That is the bridge between product requirements and runtime design.

In [ ]:
workloads = pd.DataFrame([
    {"workload": "chat_assistant", "prompt_tokens": 900, "output_tokens": 180, "concurrency_target": 6, "reliability_target": 0.995},
    {"workload": "retrieval_qa", "prompt_tokens": 2800, "output_tokens": 120, "concurrency_target": 4, "reliability_target": 0.995},
    {"workload": "code_helper", "prompt_tokens": 1800, "output_tokens": 320, "concurrency_target": 5, "reliability_target": 0.99},
    {"workload": "batch_summarizer", "prompt_tokens": 6400, "output_tokens": 220, "concurrency_target": 2, "reliability_target": 0.98},
    {"workload": "schema_extractor", "prompt_tokens": 1400, "output_tokens": 80, "concurrency_target": 10, "reliability_target": 0.999},
])
workloads["total_tokens"] = workloads["prompt_tokens"] + workloads["output_tokens"]
workloads["prompt_share"] = (workloads["prompt_tokens"] / workloads["total_tokens"]).round(3)
workloads.sort_values(["prompt_tokens", "concurrency_target"], ascending=[False, False])

## Step 3: Sketch the request lifecycle

For an open-model runtime, a request usually crosses the same stages even when the implementation differs:

- admission and queueing
- tokenization and prompt normalization
- prefill over the prompt
- decode over the generated tokens
- post-processing, validation, and logging

In [ ]:
def estimate_request(profile, prefix_hit_rate=0.0):
    prompt_tokens = profile["prompt_tokens"]
    effective_prompt_tokens = max(1.0, prompt_tokens * (1.0 - prefix_hit_rate))
    output_tokens = profile["output_tokens"]
    queue_ms = max(0.0, 12.0 * (profile["concurrency_target"] - 1))
    tokenize_ms = 6.0 + 0.012 * prompt_tokens
    prefill_ms = 18.0 + 0.03 * (effective_prompt_tokens ** 1.12)
    decode_ms = output_tokens * (2.2 + 0.004 * prompt_tokens)
    postprocess_ms = 8.0 + 0.02 * output_tokens
    total_ms = queue_ms + tokenize_ms + prefill_ms + decode_ms + postprocess_ms
    return [
        {"stage": "queue", "ms": round(queue_ms, 2)},
        {"stage": "tokenize", "ms": round(tokenize_ms, 2)},
        {"stage": "prefill", "ms": round(prefill_ms, 2)},
        {"stage": "decode", "ms": round(decode_ms, 2)},
        {"stage": "postprocess", "ms": round(postprocess_ms, 2)},
        {"stage": "total", "ms": round(total_ms, 2)},
    ]

In [ ]:
lifecycle_rows = []
for workload in workloads.to_dict("records"):
    for stage in estimate_request(workload, prefix_hit_rate=0.35):
        if stage["stage"] != "total":
            lifecycle_rows.append({"workload": workload["workload"], **stage})

lifecycle = pd.DataFrame(lifecycle_rows)
lifecycle

## Step 4: Visualize where the time goes

The waterfall below is synthetic, but it exposes an important pattern: long prompts grow prefill cost first, while long answers grow decode cost first.

In [ ]:
pivot = lifecycle.pivot(index="workload", columns="stage", values="ms").fillna(0.0)
stage_order = ["queue", "tokenize", "prefill", "decode", "postprocess"]
colors = {
    "queue": "#6c757d",
    "tokenize": "#4c78a8",
    "prefill": "#f58518",
    "decode": "#54a24b",
    "postprocess": "#e45756",
}
fig, ax = plt.subplots(figsize=(10, 5))
running = np.zeros(len(pivot))
for stage in stage_order:
    values = pivot[stage].to_numpy()
    ax.bar(pivot.index, values, bottom=running, label=stage, color=colors[stage])
    running = running + values
ax.set_ylabel("milliseconds")
ax.set_title("Synthetic request waterfall by workload")
ax.legend(loc="upper right")
plt.xticks(rotation=20)
plt.show()

## Step 5: Prefill and decode are different jobs

Prefill digests the prompt. Decode emits one token step at a time while carrying the growing context forward. That means prompt length and output length change different parts of the runtime.

In [ ]:
def phase_cost(prompt_tokens, output_tokens, prefix_hit_rate=0.0):
    effective_prompt_tokens = max(1.0, prompt_tokens * (1.0 - prefix_hit_rate))
    prefill_ms = 20.0 + 0.028 * (effective_prompt_tokens ** 1.14)
    decode_per_token_ms = 2.0 + 0.0035 * prompt_tokens
    decode_ms = output_tokens * decode_per_token_ms
    total_ms = prefill_ms + decode_ms
    return {
        "prefill_ms": round(prefill_ms, 2),
        "decode_ms": round(decode_ms, 2),
        "decode_per_token_ms": round(decode_per_token_ms, 3),
        "total_ms": round(total_ms, 2),
        "prefill_share": round(prefill_ms / total_ms, 3),
    }

phase_rows = []
for prompt_tokens in [64, 256, 1024, 4096, 8192]:
    for output_tokens in [32, 128, 256]:
        metrics = phase_cost(prompt_tokens, output_tokens, prefix_hit_rate=0.25)
        phase_rows.append({"prompt_tokens": prompt_tokens, "output_tokens": output_tokens, **metrics})

phase_df = pd.DataFrame(phase_rows)
phase_df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for output_tokens, frame in phase_df.groupby("output_tokens"):
    frame = frame.sort_values("prompt_tokens")
    ax.plot(frame["prompt_tokens"], frame["prefill_share"], marker="o", label=f"output={output_tokens}")
ax.set_xscale("log", base=2)
ax.set_xlabel("prompt tokens")
ax.set_ylabel("prefill share of total latency")
ax.set_title("Longer prompts push more latency into prefill")
ax.legend()
plt.show()

## Step 6: Queueing changes the product experience

A runtime can look fast in isolation and still feel slow in production because requests wait their turn. Queueing is why throughput engineering matters.

In [ ]:
def simulate_queue(arrival_rate_rps, service_ms, n_requests=250, seed=0):
    rng = random.Random(seed)
    rows = []
    arrival_clock = 0.0
    server_free_ms = 0.0
    for request_id in range(n_requests):
        interarrival_ms = rng.expovariate(arrival_rate_rps) * 1000.0
        arrival_clock += interarrival_ms
        start_ms = max(arrival_clock, server_free_ms)
        wait_ms = start_ms - arrival_clock
        realized_service_ms = service_ms * rng.uniform(0.88, 1.18)
        finish_ms = start_ms + realized_service_ms
        server_free_ms = finish_ms
        rows.append({
            "request_id": request_id,
            "arrival_ms": round(arrival_clock, 2),
            "wait_ms": round(wait_ms, 2),
            "service_ms": round(realized_service_ms, 2),
            "latency_ms": round(wait_ms + realized_service_ms, 2),
        })
    return pd.DataFrame(rows)

In [ ]:
queue_runs = []
for arrival_rate in [1.0, 2.0, 3.0, 4.0]:
    run = simulate_queue(arrival_rate_rps=arrival_rate, service_ms=280.0, n_requests=300, seed=int(arrival_rate * 10))
    run["arrival_rate_rps"] = arrival_rate
    queue_runs.append(run)

queue_runs = pd.concat(queue_runs, ignore_index=True)
queue_summary = queue_runs.groupby("arrival_rate_rps").agg(
    avg_wait_ms=("wait_ms", "mean"),
    p95_wait_ms=("wait_ms", lambda values: np.percentile(values, 95)),
    avg_latency_ms=("latency_ms", "mean"),
).round(2).reset_index()
queue_summary

## Step 7: Cache thinking starts with repeated prefixes

Cache reuse matters whenever many requests share the same beginning: a stable system prompt, a shared retrieval scaffold, or a common conversation history. The runtime question becomes: how much prefill can we avoid doing twice?

In [ ]:
cache_rows = []
for shared_prefix in [0, 256, 1024, 4096]:
    for reuse_count in [1, 4, 16, 64]:
        uncached = phase_cost(shared_prefix + 512, 128, prefix_hit_rate=0.0)
        cached = phase_cost(shared_prefix + 512, 128, prefix_hit_rate=shared_prefix / max(shared_prefix + 512, 1))
        saved_ms = max(0.0, uncached["total_ms"] - cached["total_ms"])
        cache_rows.append({
            "shared_prefix_tokens": shared_prefix,
            "reuse_count": reuse_count,
            "saved_ms_per_request": round(saved_ms, 2),
            "saved_ms_total": round(saved_ms * reuse_count, 2),
        })

cache_df = pd.DataFrame(cache_rows)
cache_df.sort_values(["shared_prefix_tokens", "reuse_count"]).head(12)

## Step 8: Compare candidate systems with explicit scorecards

A system design discussion is healthier when you write the trade-offs down. We will rank a few synthetic configurations using weighted metrics instead of opinions.

In [ ]:
configs = pd.DataFrame([
    {"config": "local_demo", "quality": 0.78, "p95_latency_ms": 1800, "tokens_per_second": 42, "cost_per_1k_usd": 0.08, "availability": 0.97},
    {"config": "cached_single_node", "quality": 0.83, "p95_latency_ms": 980, "tokens_per_second": 120, "cost_per_1k_usd": 0.04, "availability": 0.992},
    {"config": "throughput_optimized", "quality": 0.82, "p95_latency_ms": 760, "tokens_per_second": 180, "cost_per_1k_usd": 0.03, "availability": 0.989},
    {"config": "overcompressed_edge", "quality": 0.69, "p95_latency_ms": 620, "tokens_per_second": 65, "cost_per_1k_usd": 0.02, "availability": 0.965},
])
configs["quality_score"] = min_max_scale(configs["quality"], larger_is_better=True)
configs["latency_score"] = min_max_scale(configs["p95_latency_ms"], larger_is_better=False)
configs["throughput_score"] = min_max_scale(configs["tokens_per_second"], larger_is_better=True)
configs["cost_score"] = min_max_scale(configs["cost_per_1k_usd"], larger_is_better=False)
configs["availability_score"] = min_max_scale(configs["availability"], larger_is_better=True)
configs["composite_score"] = (
    0.30 * configs["quality_score"] +
    0.25 * configs["latency_score"] +
    0.20 * configs["throughput_score"] +
    0.10 * configs["cost_score"] +
    0.15 * configs["availability_score"]
).round(3)
scorecard = configs.sort_values(["composite_score", "quality"], ascending=[False, False]).reset_index(drop=True)
scorecard

## Step 9: Observe the system, not just the final text

Strong systems work logs stage timings, cache hits, validation outcomes, and retry reasons. That makes regressions explainable instead of mysterious.

In [ ]:
event_log = []
for request_id in range(12):
    profile = workloads.iloc[request_id % len(workloads)].to_dict()
    prefix_hit_rate = [0.0, 0.25, 0.5, 0.75][request_id % 4]
    for event in estimate_request(profile, prefix_hit_rate=prefix_hit_rate):
        if event["stage"] == "total":
            continue
        event_log.append({
            "request_id": request_id,
            "workload": profile["workload"],
            "stage": event["stage"],
            "ms": event["ms"],
            "prefix_hit_rate": prefix_hit_rate,
            "status": "ok" if event["stage"] != "queue" or event["ms"] < 80 else "slow",
        })

events = pd.DataFrame(event_log)
events.groupby(["stage", "status"]).agg(avg_ms=("ms", "mean"), count=("ms", "size")).round(2).reset_index()

## Step 10: Export a design checklist artifact

Small artifacts make later notebooks reproducible. We will save a checklist that names the questions a production-minded team should answer before changing runtimes or deployment tiers.

In [ ]:
design_checklist = {
    "workload_questions": [
        "What are the realistic prompt and completion length ranges?",
        "Which requests share prefixes that could hit a cache?",
        "What is the concurrency target at the 95th percentile hour?",
    ],
    "latency_questions": [
        "What are the targets for time to first token and end-to-end latency?",
        "How much of the current latency is queueing rather than compute?",
    ],
    "memory_questions": [
        "How much memory is left after weights load?",
        "How large can the KV cache grow before backpressure is needed?",
    ],
    "reliability_questions": [
        "What should happen when the queue is full?",
        "Which metrics and logs would explain a sudden slowdown?",
    ],
}

checklist_path = NOTEBOOK_ROOT / "design_checklist.json"
with checklist_path.open("w", encoding="utf-8") as handle:
    json.dump(design_checklist, handle, indent=2)

print("Saved:", checklist_path.resolve())

## Exercises

1. Increase the prompt length for `retrieval_qa` and observe which stage grows fastest.
2. Change the arrival rate in the queue simulator until p95 wait time becomes larger than service time.
3. Add a second scorecard weighting that values quality more than latency and compare the ranking.

In [ ]:
student_brief = pd.DataFrame([
    {"question": "Which workload is most prefill-heavy?", "your_answer": ""},
    {"question": "Which workload is most queue-sensitive?", "your_answer": ""},
    {"question": "Where would prefix caching help most?", "your_answer": ""},
])
student_brief

## Recap

Module 1 starts with mental models before concrete runtimes. You now have a vocabulary for request stages, prefill versus decode, queueing, cache reuse, and scorecards. That vocabulary will make the later runtime notebooks legible.

In [ ]:
assert phase_cost(4096, 128)["prefill_ms"] > phase_cost(256, 128)["prefill_ms"]
assert queue_summary["p95_wait_ms"].max() > queue_summary["p95_wait_ms"].min()
assert scorecard.iloc[0]["composite_score"] >= scorecard.iloc[-1]["composite_score"]
print("Notebook 01 sanity checks passed.")